4.3 PDF 파일 다루기
pdf 연구보고서 로드하고 인덱싱 방법 실행

4.3.1 데이터 준비
[pdf 로드하는 방법]
1. SimpleDirectoryReader 라이브러리 활용
2. GoogleDocsReader
3. NotionPageReader
4. SlacReader

In [5]:
from llama_index.core import SimpleDirectoryReader

#input_files : 지정한 디렉토리 내 지정한 파일만 로드
#input_dir : 지정한 디렉토리에 있는 모든 파일 로드
reader = SimpleDirectoryReader(input_files=["data/인문학의 새로운 역할에 대한 고찰.pdf"])
documents = reader.load_data()

print(len(documents))

137


4.3.2 텍스트 분할
청킹(chunking) : 긴 문장을 짧게 나누어 노드에 담는 작업
하나의 청크는 단일한 의미를 담고 있어야 RAG 성능이 좋아짐

1. 토큰 단위 분할(token text splitting)
  - TokenTextSplitter
2. 의미 단위 분할(semantic splitting)
  - SemanticSplitterNodeParser
3. 문장 단위 분할(sentence splitting)
  - SentenceSplitter

In [7]:
#1.토큰 단위 분할
from llama_index.core.node_parser import TokenTextSplitter

#TokenTextSplitter 설정
splitter = TokenTextSplitter(chunk_size=1024, chunk_overlap=20)

#TokenTextSplitter 적용 후 노드에 담기
nodes = splitter.get_nodes_from_documents(documents)

print(len(nodes))

186


In [9]:
#2.의미 단위 분할
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.openai import OpenAIEmbedding

#embed_model 인스턴스 만들기
embed_model = OpenAIEmbedding()

#semantic_splitter 설정하기
semantic_splitter = SemanticSplitterNodeParser(
    buffer_size=10,
    breakpoint_percentile_threshold=95,
    embed_model=embed_model
)

#semantic_splitter 적용 후 노드에 담기
nodes_semantic=semantic_splitter.get_nodes_from_documents(documents)
print(len(nodes_semantic))

199


In [10]:
#3.문장 단위 분할
from llama_index.core.node_parser import SentenceSplitter

#semantic splitter 설정하기
splitter = SentenceSplitter(
    chunk_size=1024,
    chunk_overlap=20,
)

#semantic_splitter 적용 후 노드에 담기
nodes_sentence = splitter.get_nodes_from_documents(documents)

print(len(nodes_sentence))

185


4.3.3 인덱싱
데이터를 구조화하여 빠르게 검색할 수 있도록 하는 과정
- 벡터 스토어 인덱스: 각 노드의 텍스트를 임제딩으로 변환하고, 쿼리가 들어오면 쿼리 또한 벡터 임베딩으로 변환한 뒤, 이 두 임베딩 간의 유사도를 계산해 가장 유사한 상위 K개의 노드 반환

In [12]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex(nodes_sentence)
print(index)

4.3.4 쿼리 실행

In [14]:
query_engine = index.as_query_engine(similarity_top_k=5)
response = query_engine.query(
    "한국어로만 답해줘"
    "디지털 인문학은 인문학을 어떤 방식과 관점으로 다루는 학문인지 알려줘"
)

print(response)

디지털 인문학은 전통적인 인문학의 방법과 관점을 디지털 기술과 데이터에 적용하여 인문학적인 주제를 연구하고 이해하는 학문입니다. 이를 통해 디지털 인문학은 과거의 문헌 자료를 디지털화하고 분석하는 것뿐만 아니라, 새로운 연구 방법과 도구를 활용하여 인문학적인 문제를 탐구하고 해석하는 새로운 접근 방식을 제시하고 있습니다.


4.4 텍스트 파일 다루기
- SimpleDirectoryReader
temperature : LLM이 생성하는 답변의 랜덤성 조절, 0~1의 값, 0=예측 가능한 답변, 1=다양하고 창의적인 답변 

In [15]:
from llama_index.core import SimpleDirectoryReader

reader = SimpleDirectoryReader(input_files=["data/paul_graham_essay.txt"])
documents = reader.load_data()

print(len(documents))

1


In [16]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

llm = OpenAI(model="gpt-4o", temperature=0.2)

#노드 생성
from llama_index.core.node_parser import SemanticSplitterNodeParser

semantic_splitter = SemanticSplitterNodeParser(
    buffer_size=10,
    breakpoint_percentile_threshold=80,
    embed_model=embed_model
)

txt_nodes_semantic = semantic_splitter.get_nodes_from_documents(documents)

print("노드 개수:", len(txt_nodes_semantic))
print(txt_nodes_semantic[10].get_content())

노드 개수: 152
It was not, in fact, simply a matter of teaching SHRDLU more words. That whole way of doing AI, with explicit data structures representing concepts, was not going to work. Its brokenness did, as so often happens, generate a lot of opportunities to write papers about various band-aids that could be applied to it, but it was never going to get us Mike.

So I looked around to see what I could salvage from the wreckage of my plans, and there was Lisp. 


In [17]:
#인덱스 생성
from llama_index.core import VectorStoreIndex

txt_index = VectorStoreIndex(txt_nodes_semantic)

In [19]:
#쿼리 실행
txt_query_engine = txt_index.as_query_engine(similarity_top_k=5)

#쿼리 실행하기
txt_response = txt_query_engine.query(
    "한국어로만 답해줘"
    "저자는 유년 시절에 무엇에 열정을 쏟았어? 자세히 설명해줘"
)

print(txt_response)

저자는 유년 시절에 프로그래밍에 열정을 쏟았습니다. 간단한 게임을 만들고, 모델 로켓이 얼마나 높이 날아갈지 예측하는 프로그램을 작성했으며, 아버지가 책을 쓸 때 사용한 워드 프로세서도 만들었습니다. 프로그래밍을 좋아했지만 대학에서는 철학을 공부할 계획이었으나, 대학에 진학하면서 철학이 기대했던 것보다 덜 흥미로운 것으로 나타났습니다. 이후 철학 수업을 계속 들었지만 지루했고, 다른 분야에서는 무시해도 되는 가장자리 케이스들만 남았다고 느꼈습니다.


4.4.2 인덱스저장 : 크로마 사용하기
- 기본 RAG 실습 : 텍스트 파일 노드 -> 인덱싱 -> 질의/응답 실행
  단점 : 문서를 새로 로드하고 임베딩 생성함 -> 자원 낭비
    -> "벡터 저장소" 사용하면 해결됨
- 벡터 저장소 : 한번 생성된 임베딩을 저장해두고 필요할때마다 불러서 사용

In [ ]:
#임베딩 저장하기
import chromadb
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

documents = SimpleDirectoryReader(input_files=["data/paul_graham_essay.txt"]).load_data()
db = chromadb.PersistentClient(path="./chroma_db_test")

In [ ]:
chroma_collection = db.get_or_create_collection("quickstart_v.1")
#벡터 스토어 설정
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storageContext = StorageContext.from_defaults(vector_store=vector_store)

#인덱스 구축
index = VectorStoreIndex.from_documents(
    documents, storage_context=storageContext
)

query_engine = index.as_query_engine()
response = query_engine.query(
     "한국어로만 답해줘"
     "저자는 유년 시절에 어떤 작업에 열중했어? 자세히 설명해줘"
)
print(response)

유년 시절에 저자는 글쓰기와 프로그래밍에 주력했습니다. 학교 외에서 주로 글쓰기와 프로그래밍을 하며, 9학년 때 IBM 1401 컴퓨터를 사용하여 프로그래밍을 시도했습니다. 초기에는 포트란 언어를 사용하였고, 프로그램을 펀칭된 카드에 입력한 후 이를 카드 리더에 넣어 실행시켰습니다. 이후에는 마이크로컴퓨터가 등장하면서 직접 컴퓨터에 프로그램을 입력할 수 있게 되었고, TRS-80와 Apple II와 같은 컴퓨터를 사용하며 게임을 만들고 워드 프로세서를 개발하는 등 프로그래밍에 열중했습니다.


In [23]:
#임베딩 불러오기

#크로마 클라이언트 초기화
db = chromadb.PersistentClient(path="./chroma_db_test")

#컬렉션 호출
chroma_collection = db.get_or_create_collection("quickstart_v.1")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storageContext = StorageContext.from_defaults(vector_store=vector_store)

#저장했던 인덱스 로드
index = VectorStoreIndex.from_vector_store(
    vector_store, storage_context=storageContext
)

query_engine = index.as_query_engine()
response = query_engine.query(
     "한국어로만 답해줘"
     "저자에게 프로그래밍은 어떤 의미인가요?"
)
print(response)

프로그래밍은 저자에게 컴퓨터와 상호작용하며 무엇인가를 만들어내는 창조적인 활동으로 보입니다. 처음에는 IBM 1401에서의 프로그래밍 경험을 통해 혼란스러웠지만, 이후에는 개인용 컴퓨터를 통해 직접 프로그램을 작성하며 게임이나 예측 프로그램, 워드 프로세서 등을 개발하면서 프로그래밍에 대한 열정을 키웠습니다. 이를 통해 프로그래밍은 그의 창의성을 펼칠 수 있는 수단으로서 중요한 의미를 갖게 되었습니다.


4.5 CSV 파일 다루기
- SimpleDirectoryReader : CSV 파일을 로드 -> 헤더를 읽지 못함
    -> CSVReader와 함께 사용

In [ ]:
from llama_index.core import SimpleDirectoryReader

reader = SimpleDirectoryReader(input_files=["data/범죄 발생 장소별 통계_2023.csv"])
documents = reader.load_data()

print("documents 개수:", len(documents))
print(documents[0])

#결과 : 헤더를 읽지 못함

documents 개수: 1
Doc ID: c3111e9e-965f-4b41-9a08-67c785fc5cef
Text: 강력범죄,살인미수,48,66,38,16,6,0,1,4,8,0,0,0,0,0,2,0,2,17,0,0,0,0,2,1,4
,3,0,0,0,0,0,0,0,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8,1,7,2,2,0,0,0,0,1
,29,4 강력범죄,살인미수등,61,67,45,19,13,1,1,16,40,0,1,3,1,2,0,0,12,13,3,1,1,0,
14,1,10,6,1,1,0,0,0,0,1,5,0,0,3,0,0,0,1,0,0,1,0,3,0,0,1,1,14,3,15,1,3,
0,3,0,3,1,93,7
강력범죄,강도,36,40,29,42,9,1,1,18,42,3,4,13,35,3,0,3,23,45,1,1,...


In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.readers.file import CSVReader

parser = CSVReader()
file_extractor = {".csv":parser}

documents = SimpleDirectoryReader(
    input_files=["data/범죄 발생 장소별 통계_2023.csv"],
    file_extractor=file_extractor
).load_data()

print("documents 개수:", len(documents))
print(documents[0])

#결과 : 헤더를 제대로 읽음

documents 개수: 1
Doc ID: f484fc8c-6060-41ad-a958-258c580403ed
Text: ﻿"범죄대분류, 범죄중분류, 단독주택_다가구_다중, 아파트, 다세대_연립, 오피스텔_원룸, 기타거주시설_기숙사 등,
고속도로, 자동차 전용도로, 일반도로, 통행로_보도_골목길, 백화점, 대형할인점, 슈퍼마켓_소매점, 편의점, 시장_노점,
창고_매장창고한정, 무인상점, 기타상점, 숙박업소_호텔_모텔_여관, 목욕탕_찜질방_사우나, 이발소_미용실, 마사지업소,
공중위생업소_기타, 음식점, 카페, 주점, 단란_유흥주점_나이트_클럽_카바레, 버스터미널_정류소, 지하철역_전철역, 기차역,
여객선터미널, 공항, 버스, 택시, 자가용자동차, 지하철_전철, 기차, 선박, 비행기, 교통수단내_기타, 공연장_극장,
체육시설, 공원_놀...


In [26]:
#의미 분할 기법 사용해 document 분할, 결과는 노드 형태로 담기
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.openai import OpenAIEmbedding

embed_model = OpenAIEmbedding()
semantic_splitter = SemanticSplitterNodeParser(
    buffer_size=10,
    breakpoint_percentile_threshold=80,
    embed_model=embed_model
)

csv_nodes_semantic = semantic_splitter.get_nodes_from_documents(documents)

print("노드 개수:", len(csv_nodes_semantic))
print(csv_nodes_semantic[0].get_content())

노드 개수: 1
﻿"범죄대분류, 범죄중분류, 단독주택_다가구_다중, 아파트, 다세대_연립, 오피스텔_원룸, 기타거주시설_기숙사 등, 고속도로, 자동차 전용도로, 일반도로, 통행로_보도_골목길, 백화점, 대형할인점, 슈퍼마켓_소매점, 편의점, 시장_노점, 창고_매장창고한정, 무인상점, 기타상점, 숙박업소_호텔_모텔_여관, 목욕탕_찜질방_사우나, 이발소_미용실, 마사지업소, 공중위생업소_기타, 음식점, 카페, 주점, 단란_유흥주점_나이트_클럽_카바레, 버스터미널_정류소, 지하철역_전철역, 기차역, 여객선터미널, 공항, 버스, 택시, 자가용자동차, 지하철_전철, 기차, 선박, 비행기, 교통수단내_기타, 공연장_극장, 체육시설, 공원_놀이시설, 게임장_(PC)방, 기타_(DVD)방_유원지, 어린이집_유치원, 학교, 도서관, 학원, 기타 교육시설, 금융보험기관, 의료기관, 종교시설, 야외_산야, 해안, 폐가_공터, 공중화장실, 관공서, 군사기지_군사시설, 구금장소, 사회복지시설, 기타, 미상"
강력범죄,살인미수,48,66,38,16,6,0,1,4,8,0,0,0,0,0,2,0,2,17,0,0,0,0,2,1,4,3,0,0,0,0,0,0,0,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8,1,7,2,2,0,0,0,0,1,29,4
강력범죄,살인미수등,61,67,45,19,13,1,1,16,40,0,1,3,1,2,0,0,12,13,3,1,1,0,14,1,10,6,1,1,0,0,0,0,1,5,0,0,3,0,0,0,1,0,0,1,0,3,0,0,1,1,14,3,15,1,3,0,3,0,3,1,93,7
강력범죄,강도,36,40,29,42,9,1,1,18,42,3,4,13,35,3,0,3,23,45,1,1,7,1,7,9,10,15,0,3,0,0,0,0,3,10,0,0,0,0,0,1,1,5,7,1,1,2,0,0,0,12,1,1,19,0,0,3,1,0,0,1,100,3
강력범죄,강간,418,680,624,693,69,0,0,20,34,2,3,0,5,2,0,0,28,1592,3

In [ ]:
#인덱스 생성
from llama_index.core import VectorStoreIndex

csv_index = VectorStoreIndex(csv_nodes_semantic)
csv_query_engine = csv_index.as_query_engine(similarity_top_k=5)
csv_response = csv_query_engine.query("절도범죄가 가장 많이 일어난 장소는 어디야? 수치도 알려줘")
print(csv_response)

절도범죄가 가장 많이 일어난 장소는 "도로"이며, 해당 장소에서의 절도범죄 발생 수는 15,401건입니다.


In [28]:
csv_response = csv_query_engine.query("편의점에서는 자주 벌어지는 범죄는 어떤 것들이 있어? 그 중에서 가장 빈번하게 일어난 범죄와 범죄 건수도 알려줘")
print(csv_response)

편의점에서 자주 발생하는 범죄는 절도범죄, 폭행, 사기 등이 있습니다. 그 중에서 가장 빈번하게 발생하는 범죄는 절도범죄이며, 해당 범죄는 총 36,517건으로 가장 많이 발생한 것으로 나타났습니다.


4.6 HWP파일 다루기
'.hwp'인 문서는 다른 나라에서는 거의 사용하지 않기 때문에 데이터 로드 다소 번거로움
1. HWPReader
2. SimpleDirectoryReader

In [ ]:
#hWPReader 클래스 이용
from llama_index.readers.file import HWPReader

reader = HWPReader()
documents = reader.load_data("data/미국의 통화정책 변화가 외국자본 유출에 미치는 영향_20180528.hwp")

print("도큐먼트 개수:", len(documents))
print(documents[0])

도큐먼트 개수: 1
Doc ID: 3eebcf74-0daf-405a-af43-a81f2209afb2
Text: 汤捯    捤獥     氠瑢        漠杳        漠杳        漠杳        漠杳    
2018년 5월 28일(월) 조간   漠杳       최우진 KDI 거시경제연구부 연구위원  (044-550-4053,
wooj.choi@kdi.re.kr)   漠杳       2018년 5월 25일(금) 09:00   漠杳       KDI
홍보팀(044-550-4030, press@kdi.re.kr)   漠杳       桤灧      미국의 통화정책 변화가
외국자본 유출에 미치는 영향 漠杳       최우진 거시경제연구부 연구위원   氠瑢     湯湷      본고는 2018년
상반기...


In [4]:
#SimpleDirectoryReader 클래스 이용
from llama_index.core import SimpleDirectoryReader

reader = SimpleDirectoryReader(input_files=["data/미국의 통화정책 변화가 외국자본 유출에 미치는 영향_20180528.hwp"], encoding='euc-kr')
documents = reader.load_data()

print("도큐먼트 개수:", len(documents))
print(documents[0].get_content())

도큐먼트 개수: 1
汤捯    捤獥    氠瑢    
漠杳    
漠杳    
漠杳    
漠杳    
2018년 5월 28일(월) 조간
漠杳    
최우진 KDI 거시경제연구부 연구위원
(044-550-4053, wooj.choi@kdi.re.kr)
漠杳    
2018년 5월 25일(금) 09:00
漠杳    
KDI 홍보팀(044-550-4030, press@kdi.re.kr)
漠杳    
桤灧    
미국의 통화정책 변화가 
외국자본 유출에 미치는 영향漠杳    
최우진 거시경제연구부 연구위원
氠瑢    湯湷    
본고는 2018년 상반기 『KDI 경제전망』에 수록될 예정임.

汤捯    捤獥    湯湷    漠杳    연구위원 최우진
汤捯    미국의 통화정책 변화가 외국자본 유출에 미치는 영향
1. 문제제기潴景    潴景    慤桥    慤桥    桤灧    
漠杳    漠杳    
湯慴    
漠杳    漠杳    
湯慴    
漠杳    漠杳    
미국의 통화정책 변화가 외국자본 유출에 미치는 영향
漠杳    漠杳    
미국의 통화정책 변화가 외국자본 유출에 미치는 영향
漠杳     미국의 금리인상이 본격화될 것으로 예상되는 가운데, 신흥국 불안에 따른 국제금융시장의 변동성도 확대되고 있어 급격한 외국자본 유출에 대한 우려가 확대됨.
漠杳     2008년 금융위기 이후 ‘제로금리’를 유지하였던 미국의 통화당국은 2015년 12월을 시작으로 현재까지 정책금리를 25bp씩 여섯 차례 인상한 바 있음.
漠杳     최근에는 아르헨티나가 자국 경제불안에 따른 대외신뢰도 저하 및 통화가치 하락으로 IMF에 구제금융을 신청하면서 미국의 금리인상에 따른 신흥국의 외국자본 유출 우려가 높아지는 상황
漠杳     외국자본 유출은 형태에 따라 실물경제에 미치는 영향이 다르게 나타날 수 있음.
漠杳     서브프라임 

In [5]:
#LLM과 임베딩 모델 설정
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

llm = OpenAI(model="gpt-4o", temperature=0.2)
embed_model = OpenAIEmbedding()

Settings.llm = llm
Settings.embed_model = embed_model

#노드생성
from llama_index.core.node_parser import SemanticSplitterNodeParser

semantic_splitter = SemanticSplitterNodeParser(
    buffer_size=19,
    breakpoint_percentile_threshold=80,
    embed_model=embed_model
)
hwp_nodes_semantic = semantic_splitter.get_nodes_from_documents(documents)
print("노드의개수:", len(hwp_nodes_semantic))
print(hwp_nodes_semantic[11].get_content())

노드의개수: 13
漠杳     현재 우리 경제는 3,984억달러 규모의 외환보유액을 보유하고 있는바, 이는 단기채무의 3.2배의 규모로서 통상적인 수준을 넘어서는 금리인상 충격에 따른 자본유출을 충분히 감내할 수 있는 수준
漠杳     한편, 국제금융시장의 변동성(VIX)이 증가하는 경우 차입투자 자금을 중심으로 외국자본 유출에 영향을 미칠 가능성이 있음.
漠杳     특히, 최근 신흥국 전반에서 외국자본 유출 가능성이 확대되고 있다는 점을 고려할 때 변동성이 급격히 확대되는 상황에 대한 면밀한 모니터링을 요함.



In [6]:
#인덱스 생성
from llama_index.core import VectorStoreIndex

hwp_index = VectorStoreIndex(hwp_nodes_semantic)
hwp_query_engine = hwp_index.as_query_engine(similarity_top_k=5)
hwp_response = hwp_query_engine.query("미국 금리가 올라가면 한국 금융 시장은 어떤 영향을 받아? 한국어로 정리해줘")
print(hwp_response)

미국 금리가 인상되면 한국 금융 시장에서는 주로 부채성 자금, 즉 채권 및 차입 투자를 중심으로 외국자본이 유출될 가능성이 있습니다. 그러나 이러한 유출의 규모는 한국 경제의 규모와 외환보유액 등을 고려할 때 미미한 수준으로 평가됩니다. 미국 금리 인상은 외국자본 유출을 유발할 수 있지만, 그 영향은 제한적이며 전체 외국자본의 유입 기조는 유지될 수 있습니다.


In [7]:
print("소스노드의수:", len(hwp_response.source_nodes))
print(hwp_response.source_nodes[0].get_content())

소스노드의수: 5
漠杳     1년 만기 국공채의 시장금리(한국 통화 안정증권 및 미 재무부 채권)의 차이를 분석
漠杳     한미 금리차(한국 금리–미국 금리)와 외국자본 유출의 추이를 살펴보면, 금리차가 확대되는 시기에 외국자본이 오히려 유출되는 흐름을 나타내었던 것으로 보이나 계수 추정치가 통계적으로 유의하지 않음.
╺ 이는 미국 금리의 인상이 오히려 외국자본의 유입을 가져온다고 해석될 수 있어 이자율 평형설로써 예상되는 결과와는 다른 모습
漠杳     한편, 글로벌 금융위기 시기를 통제한 경우에는 외국자본이 유입되는 결과를 보이나 역시 계수 추정치가 통계적으로 유의하지 않음.
한미 금리차와 외국자본 유출의 관계
漠杳      氠瑢    
(3)
(4)
총투자
총투자
한미 금리차
(한국–미국)
0.01
(0.08)
-0.03
(-0.32)
글로벌 
금융위기 더미
4.60***
(8.39)
N
60
60
R-sq
0.48
0.64
           주: 회귀분석에서는 실질 GDP 성장률을 통제하였음.
         자료: 한국은행; Federal Reserve Bank of St. Louis; Bloomberg.
漠杳     한편, 한국과 미국 금융시장의 구조적 차이를 반영하기 위해 모형을 재설정하여 분석을 시도한 결과에서도, 미국의 금리인상이 외국자본의 유출에 미치는 영향은 미미한 것으로 나타남.

